# Spur Review: named Deno + Python acceptance evaluation

This notebook evaluates the current runtime against the captured DuckDB PR #28731 review design. It does not implement or release a native app. Each experiment explicitly selects its saved profile; notebook defaults remain factory.

The fixture pins base `629e395a854d72dcb1f414e1a0451c68ac0e9f1f` and head `d29b26bb92ac940a0807be0f43cc385b31623839`. No GitHub refresh, dependency installation or full repository re-index occurs here. Live named runtime behavior and captured domain evidence are reported separately.

In [ ]:
const cfg = await import('@review/review_config.ts');
const fixture = JSON.parse(await Deno.readTextFile('design-data.json'));
if (fixture.head !== cfg.expectedHead) throw Error('Review snapshot mismatch');
console.log(JSON.stringify({profile:'review-ui',label:cfg.label,cwd:Deno.cwd(),pid:Deno.pid,deno:Deno.version,head:fixture.head,changed_files:fixture.files.length,spur_type:typeof spur,put_type:typeof spur?.put,get_type:typeof spur?.get,anywidget_type:typeof spur?.anywidget}));

In [2]:
import sys, os, json, hashlib
sys.dont_write_bytecode = True
from review_config import FIXTURE_PATH, EXPECTED_HEAD
import duckdb
review_data = json.loads(FIXTURE_PATH.read_text())
assert review_data["head"] == EXPECTED_HEAD
assert hashlib.sha256(FIXTURE_PATH.read_bytes()).hexdigest() == "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2"
assert len(review_data["files"]) == 5
review_db = duckdb.connect()
review_db.execute("CREATE TABLE changed_files AS SELECT unnest(files) AS file FROM read_json_auto(?)", [str(FIXTURE_PATH)])
review_counts = review_db.sql("SELECT count(*) FROM changed_files").fetchone()[0]
assert review_counts == 5
review_refs = [r["evidence"] for c in review_data["contexts"].values() for r in c["rows"]]
assert all(ref in review_data["evidence"] for ref in review_refs)
assert any("Unchanged" in n["state"] for n in review_data["contexts"]["incoming"]["nodes"])
assert all(not p["formal_owner_verified"] for p in review_data["people"])
review_observation = {
 "profile":"review-analysis","pid":os.getpid(),"cwd":os.getcwd(),"executable":sys.executable,
 "duckdb_runtime":duckdb.__version__,"head":review_data["head"],"base":review_data["base"],
 "fixture_sha256":hashlib.sha256(FIXTURE_PATH.read_bytes()).hexdigest(),
 "changed_files_sql":review_counts,"review_questions":len(review_data["contexts"]),
 "source_evidence":len(review_data["evidence"]),"components":len(review_data["components"]),
 "historical_people_candidates":len(review_data["people"]),
 "resolved_context_row_refs":len(review_refs),"all_context_row_refs_exist":True,
 "unchanged_context_retained":True,"formal_ownership_verified":False,
 "installed_review_extensions":review_db.sql("SELECT extension_name, installed, loaded FROM duckdb_extensions() WHERE extension_name IN ('spur_graph','duck_tails','duckpgq')").fetchall(),
 "spur_in_globals":"spur" in globals()
}
print(json.dumps(review_observation,indent=2))

{
  "profile": "review-analysis",
  "pid": 24991,
  "cwd": "/private/var/folders/mv/vtzfmys92yl9rwcsgf9n1x780000gn/T/spur-review-named-kibzfbbp",
  "executable": "/Users/kevintruong/.spur/jupyter/venv/bin/python",
  "duckdb_runtime": "1.5.5",
  "head": "d29b26bb92ac940a0807be0f43cc385b31623839",
  "base": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f",
  "fixture_sha256": "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2",
  "changed_files_sql": 5,
  "review_questions": 4,
  "source_evidence": 21,
  "components": 6,
  "historical_people_candidates": 7,
  "resolved_context_row_refs": 12,
  "all_context_row_refs_exist": true,
  "unchanged_context_retained": true,
  "formal_ownership_verified": false,
  "installed_review_extensions": [
    [
      "duck_tails",
      true,
      false
    ]
  ],
  "spur_in_globals": false
}


In [3]:
# Observe the named-profile bridge explicitly; this is not a successful port publication.
import json
bridge_probe = {'profile':'review-analysis','spur_in_globals':'spur' in globals(),'published':False}
try:
    spur.put('review_run_probe', [{'head':review_data['head'],'files':len(review_data['files'])}])
    bridge_probe['published'] = True
except Exception as exc:
    bridge_probe['exception_type'] = type(exc).__name__
    bridge_probe['message'] = str(exc)
assert bridge_probe['published'] is False, 'Expected disabled named Python ports contract to stay explicit'
print(json.dumps(bridge_probe, indent=2))

{
  "profile": "review-analysis",
  "spur_in_globals": false,
  "published": false,
  "exception_type": "NameError",
  "message": "name 'spur' is not defined"
}


In [2]:
// Standalone control only. This does not satisfy the named Deno acceptance test.
const html = await Deno.readTextFile('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/prototype.html');
const digest = [...new Uint8Array(await crypto.subtle.digest('SHA-256',new TextEncoder().encode(html)))].map(x=>x.toString(16).padStart(2,'0')).join('');
console.log(JSON.stringify({track:'standalone captured HTML control',deno:Deno.version,pid:Deno.pid,bytes:new TextEncoder().encode(html).length,sha256:digest,live_data:false,native_app:false}));
await Deno.jupyter.display({'text/html':html},{raw:true});

{"track":"standalone captured HTML control","deno":{"deno":"2.8.1","v8":"14.9.207.2-rusty","typescript":"6.0.3"},"pid":26392,"bytes":116180,"sha256":"c791f0c88557de83b7946c01de2bdd3be210c7f2d7066c7aae2e8313ab16d473","live_data":false,"native_app":false}


<!doctype html> Spur Graph Review | interactive design S Spur Graph Review Review Inbox Repository Interactive design Captured evidence from PR #28731. Navigation is interactive; collection, AI execution and publication are not connected. DUCKDB / PULL REQUEST #28731 Preserve aggregate domains for proven singleton groups 629e395 → d29b26b 5 changed files Partial linking Review packet Changes + context Architecture People AI review Changed files 5 Select a change to explore repository context. relation_statistics_helper.cpp Modified ExtractAggregationStatsInternal HasUniqueGroups HasBaseColumnSource GetValuePreservingColumn first_last_any.cpp Modified stats_dependent_tests.json Modified verify_aggregate_state_export.json Modified distinct_on_stats_propagation.test Added Context includes unchanged files Tracked repository catalog. Semantic linking remains partial. Inspect coverage SELECTED CHANGE ExtractAggregationStatsInternal src/optimizer/relation_statistics/relation_statistics_helper.cpp Modified Context map Path list Source diff Incoming clients Dependencies Shared contracts Tests + config Who reaches this change? Unchanged clients, an unchanged wrapper and the modified implementation. Include candidates Captured PR neighborhood · frozen head API ref. API ref. Calls Relation manager relation_manager.cpp Unchanged Statistics extractor relation_statistics_extractor.cpp Unchanged ExtractAggregationStats Two overloads inside modified file Unchanged body ExtractAggregationStatsInternal relation_statistics_helper.cpp Modified Evidence limit. External API references are candidates. The wrapper-to-internal leg is syntax-bound. Review paths Select a path to inspect its witness Join-order client → public wrapper → changed internal The first leg references an overloaded API. The second has a retained syntax binding. Candidate Unchanged wrapper inside a modified file Identical body bytes; moved from base lines 554 / 562 to head 688 / 696. Syntax-bound Changed internal → new uniqueness proof Follow the new helper without treating syntax matching as compiler resolution. Syntax-bound Evidence Frozen head Syntax-bound Wrapper calls changed internal helper src/optimizer/relation_statistics/relation_statistics_helper.cpp Line 693 Source context anchor; exact graph path is in the retained receipt 691 // Join reconstruction has already published the selected input cardinalities. 
 692 RelationStatsExtractor extractor(context); 
 693 return ExtractAggregationStatsInternal(aggregate, child_stats, extractor, true); 
 694 } 
 695 
 696 optional<RelationStats> RelationStatisticsHelper::ExtractAggregationStats(LogicalAggregate &aggregate, 
 What this establishes Syntax binding from the retained two-file graph. The wrapper body is unchanged; compiler resolution is not claimed. Provenance and query scope Revision d29b26bb92ac940a0807be0f43cc385b31623839 Evidence ID native:wrapper Receipt native-incoming-paths.json [6] Graph identities and source anchors remain scoped to the originating experiment. Copy reference Source and path selection share this review run. Selection persists across review lenses. Solid: syntax / declarations · Dashed: candidates · Dotted: shared contracts Static design preview. Enable active content in the notebook to try navigation and evidence selection.

In [3]:
import sys, os, json, hashlib
sys.dont_write_bytecode = True
from review_config import FIXTURE_PATH, EXPECTED_HEAD
import duckdb
review_data = json.loads(FIXTURE_PATH.read_text())
assert review_data["head"] == EXPECTED_HEAD
assert hashlib.sha256(FIXTURE_PATH.read_bytes()).hexdigest() == "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2"
assert len(review_data["files"]) == 5
review_db = duckdb.connect()
review_db.execute("CREATE TABLE changed_files AS SELECT unnest(files) AS file FROM read_json_auto(?)", [str(FIXTURE_PATH)])
review_counts = review_db.sql("SELECT count(*) FROM changed_files").fetchall()[0][0]
assert review_counts == 5
review_refs = [r["evidence"] for c in review_data["contexts"].values() for r in c["rows"]]
assert all(ref in review_data["evidence"] for ref in review_refs)
assert any("Unchanged" in n["state"] for n in review_data["contexts"]["incoming"]["nodes"])
assert all(not p["formal_owner_verified"] for p in review_data["people"])
review_observation = {
 "profile":"review-duckdb2","pid":os.getpid(),"cwd":os.getcwd(),"executable":sys.executable,
 "duckdb_runtime":duckdb.__version__,"head":review_data["head"],"base":review_data["base"],
 "fixture_sha256":hashlib.sha256(FIXTURE_PATH.read_bytes()).hexdigest(),
 "changed_files_sql":review_counts,"review_questions":len(review_data["contexts"]),
 "source_evidence":len(review_data["evidence"]),"components":len(review_data["components"]),
 "historical_people_candidates":len(review_data["people"]),
 "resolved_context_row_refs":len(review_refs),"all_context_row_refs_exist":True,
 "unchanged_context_retained":True,"formal_ownership_verified":False,
 "installed_review_extensions":review_db.sql("SELECT extension_name, installed, loaded FROM duckdb_extensions() WHERE extension_name IN ('spur_graph','duck_tails','duckpgq')").fetchall(),
 "spur_in_globals":"spur" in globals()
}
print(json.dumps(review_observation,indent=2))
assert duckdb.__version__ == '2.0.0'


{
  "profile": "review-duckdb2",
  "pid": 26814,
  "cwd": "/private/var/folders/mv/vtzfmys92yl9rwcsgf9n1x780000gn/T/spur-review-named-kibzfbbp",
  "executable": "/Users/kevintruong/.spur/jupyter/venv-duckdb-2.0/bin/python",
  "duckdb_runtime": "2.0.0",
  "head": "d29b26bb92ac940a0807be0f43cc385b31623839",
  "base": "629e395a854d72dcb1f414e1a0451c68ac0e9f1f",
  "fixture_sha256": "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2",
  "changed_files_sql": 5,
  "review_questions": 4,
  "source_evidence": 21,
  "components": 6,
  "historical_people_candidates": 7,
  "resolved_context_row_refs": 12,
  "all_context_row_refs_exist": true,
  "unchanged_context_retained": true,
  "formal_ownership_verified": false,
  "installed_review_extensions": [],
  "spur_in_globals": false
}


In [4]:
# Load only the previously captured, hash-verified local binaries; no INSTALL/network.
import json,hashlib
from pathlib import Path
extension_receipt = Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-review-extensions-poc/extension-binaries.json')
extension_identities = json.loads(extension_receipt.read_text())
assert duckdb.__version__ == '2.0.0'
native_review_db = duckdb.connect(config={'allow_unsigned_extensions': True})
for item in extension_identities:
    extension_file = Path(item['path'])
    assert hashlib.sha256(extension_file.read_bytes()).hexdigest() == item['sha256'], 'Captured extension bytes changed'
    native_review_db.execute("LOAD '" + str(extension_file).replace("'", "''") + "'")
native_inventory = native_review_db.sql("SELECT extension_name,loaded,extension_version FROM duckdb_extensions() WHERE extension_name IN ('spur_graph','duck_tails','duckpgq') ORDER BY extension_name").fetchall()
assert len(native_inventory) == 3 and all(row[1] for row in native_inventory)
native_info = native_review_db.sql('SELECT spur_graph_info()').fetchall()[0][0]
probe_symbols = native_review_db.sql("SELECT spur_graph_symbols('probe.cpp','int helper() { return 1; } int caller() { return helper(); }')").fetchall()[0][0]
native_extension_observation = {
 'profile':'review-duckdb2','duckdb_runtime':duckdb.__version__,'pid':os.getpid(),
 'loaded':native_inventory,'spur_graph_info':native_info,
 'cpp_probe_symbol_count':len(probe_symbols),'all_artifact_hashes_match':True,
 'scope':'Local smoke on captured binaries. Not a rebuild, full PR recollection, or complete cross-file linkage.'
}
assert len(probe_symbols) >= 2
print(json.dumps(native_extension_observation,indent=2))


{
  "profile": "review-duckdb2",
  "duckdb_runtime": "2.0.0",
  "pid": 26814,
  "loaded": [
    [
      "duck_tails",
      true,
      "v1.9.0"
    ],
    [
      "duckpgq",
      true,
      "9c11bc192a"
    ],
    [
      "spur_graph",
      true,
      "0.1.0"
    ]
  ],
  "spur_graph_info": {
    "extension_version": "0.1.0",
    "facts_schema_version": 1,
    "duckdb_api_version": "2.0.0",
    "extractor_id": "source-sha256-v1:6ff57778ccc62a9cbd78f39549836fd9a7c91f55c3f94830443b239e8a17f2c2",
    "supported_languages": [
      "C",
      "Cpp",
      "Css",
      "Go",
      "Hcl",
      "Html",
      "Javascript",
      "Json",
      "JupyterNotebook",
      "Lua",
      "Markdown",
      "Mermaid",
      "Python",
      "Rust",
      "Shell",
      "Sql",
      "Terraform",
      "Toml",
      "Tsx",
      "TypeScript",
      "Yaml"
    ],
    "resolution_scope": "single_file"
  },
  "cpp_probe_symbol_count": 2,
  "all_artifact_hashes_match": true,
  "scope": "Local smoke on ca

In [ ]:
# Typed evidence traversal on the captured review neighborhood; graph discovery is not rerun.
node_rows = [(ctx + ':' + n['id'], n['title'], n['state'], n['evidence'])
             for ctx, graph in review_data['contexts'].items() for n in graph['nodes']]
edge_rows = [(ctx + ':' + e['a'], ctx + ':' + e['b'], e['kind'], e['evidence'])
             for ctx, graph in review_data['contexts'].items() for e in graph['edges']]
assert all(row[3] in review_data['evidence'] for row in node_rows + edge_rows)
native_review_db.execute('CREATE OR REPLACE TABLE review_nodes(id VARCHAR PRIMARY KEY,title VARCHAR,state VARCHAR,evidence VARCHAR)')
native_review_db.executemany('INSERT INTO review_nodes VALUES (?,?,?,?)', node_rows)
native_review_db.execute('CREATE OR REPLACE TABLE review_edges(src VARCHAR,dst VARCHAR,kind VARCHAR,evidence VARCHAR)')
native_review_db.executemany('INSERT INTO review_edges VALUES (?,?,?,?)', edge_rows)
native_review_db.execute("CREATE OR REPLACE TABLE syntax_edges AS SELECT * FROM review_edges WHERE kind='syntax'")
native_review_db.execute("""CREATE OR REPLACE PROPERTY GRAPH review_context
 VERTEX TABLES (review_nodes LABEL symbol)
 EDGE TABLES (review_edges SOURCE KEY(src) REFERENCES review_nodes(id)
 DESTINATION KEY(dst) REFERENCES review_nodes(id) LABEL relates)""")
native_review_db.execute("""CREATE OR REPLACE PROPERTY GRAPH review_syntax
 VERTEX TABLES (review_nodes LABEL symbol)
 EDGE TABLES (syntax_edges SOURCE KEY(src) REFERENCES review_nodes(id)
 DESTINATION KEY(dst) REFERENCES review_nodes(id) LABEL relates)""")
path_query = """SELECT * FROM GRAPH_TABLE({graph}
 MATCH p = ANY SHORTEST (a:symbol)-[e:relates]->{{1,2}}(b:symbol)
 WHERE b.id='incoming:internal'
 COLUMNS (a.id AS source,a.title AS source_title,a.state AS source_state,path_length(p) AS hops))
 ORDER BY hops,source"""
context_paths = native_review_db.sql(path_query.format(graph='review_context')).fetchall()
syntax_paths = native_review_db.sql(path_query.format(graph='review_syntax')).fetchall()
assert len(context_paths)==3 and len(syntax_paths)==1
assert syntax_paths[0][0]=='incoming:wrapper'
assert all('Unchanged' in row[2] for row in context_paths)
journey_observation = {
 'scope':'Captured neighborhood replay through DuckPGQ; not repository semantic completeness',
 'full_context_paths':context_paths,'syntax_only_paths':syntax_paths,
 'candidate_paths_hidden':len(context_paths)-len(syntax_paths),
 'architecture':{'components':len(review_data['components']),'alignment':'not_evaluated_no_policy'},
 'people':{'historical_candidates':len(review_data['people']),
           'github_logins_verified_in_capture':sum(p['github_login'] is not None for p in review_data['people']),
           'current_assignments':'not_refreshed','formal_ownership':'not_verified'},
 'ai_review':'not_run','run_head':review_data['head'],'source_evidence':len(review_data['evidence'])
}
print(json.dumps(journey_observation,indent=2))


ParserException: Parser Error: syntax error at or near "PROPERTY"

LINE 1: CREATE OR REPLACE PROPERTY GRAPH review_context
                          ^^^^^^^^

In [6]:
# Diagnostic: LOAD succeeded, so check the actual graph parser path separately.
parser_probe = {'inventory':native_review_db.sql("SELECT extension_name,loaded FROM duckdb_extensions() WHERE extension_name='duckpgq'").fetchall(), 'attempts':[]}
for statement in ["CREATE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL symbol)","CREATE OR REPLACE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL symbol)"]:
    try:
        native_review_db.execute(statement)
        parser_probe['attempts'].append({'statement':statement,'status':'ok'})
    except Exception as exc:
        parser_probe['attempts'].append({'statement':statement,'status':'failed','type':type(exc).__name__,'message':str(exc)})
parser_probe['functions'] = native_review_db.sql("SELECT function_name,parameters FROM duckdb_functions() WHERE function_name ILIKE '%pgq%' OR function_name='graph_table' ORDER BY function_name LIMIT 20").fetchall()
print(json.dumps(parser_probe,indent=2))

{
  "inventory": [
    [
      "duckpgq",
      true
    ]
  ],
  "attempts": [
    {
      "statement": "CREATE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL symbol)",
      "status": "failed",
      "type": "ParserException",
      "message": "Parser Error: syntax error at or near \"PROPERTY\"\n\nLINE 1: CREATE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL...\n               ^^^^^^^^"
    },
    {
      "statement": "CREATE OR REPLACE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL symbol)",
      "status": "failed",
      "type": "ParserException",
      "message": "Parser Error: syntax error at or near \"PROPERTY\"\n\nLINE 1: CREATE OR REPLACE PROPERTY GRAPH review_diag VERTEX TABLES (review_nodes LABEL...\n                          ^^^^^^^^"
    }
  ],
  "functions": [
    [
      "duckpgq_match",
      []
    ]
  ]
}


In [7]:
# Explicit SQL control, separate from the failed DuckPGQ parser path above.
# Rebuild the captured graph tables so this cell does not depend on a failed cell's partial work.
node_rows = [(ctx + ':' + n['id'], n['title'], n['state'], n['evidence'])
             for ctx, graph in review_data['contexts'].items() for n in graph['nodes']]
edge_rows = [(ctx + ':' + e['a'], ctx + ':' + e['b'], e['kind'], e['evidence'])
             for ctx, graph in review_data['contexts'].items() for e in graph['edges']]
assert all(row[3] in review_data['evidence'] for row in node_rows + edge_rows)
native_review_db.execute('CREATE OR REPLACE TABLE review_nodes(id VARCHAR PRIMARY KEY,title VARCHAR,state VARCHAR,evidence VARCHAR)')
native_review_db.executemany('INSERT INTO review_nodes VALUES (?,?,?,?)', node_rows)
native_review_db.execute('CREATE OR REPLACE TABLE review_edges(src VARCHAR,dst VARCHAR,kind VARCHAR,evidence VARCHAR)')
native_review_db.executemany('INSERT INTO review_edges VALUES (?,?,?,?)', edge_rows)
recursive_query = """WITH RECURSIVE allowed_edges AS (
 SELECT * FROM review_edges {filter}
), walk(source,dst,hops,witnesses) AS (
 SELECT src,dst,1,[evidence] FROM allowed_edges
 UNION ALL
 SELECT w.source,e.dst,w.hops+1,list_append(w.witnesses,e.evidence)
 FROM walk w JOIN allowed_edges e ON w.dst=e.src WHERE w.hops<2
)
SELECT w.source,n.title,n.state,w.hops,w.witnesses
FROM walk w JOIN review_nodes n ON n.id=w.source
WHERE w.dst='incoming:internal' ORDER BY w.hops,w.source"""
context_paths = native_review_db.sql(recursive_query.format(filter='')).fetchall()
syntax_paths = native_review_db.sql(recursive_query.format(filter="WHERE kind='syntax'")).fetchall()
assert len(context_paths)==3 and len(syntax_paths)==1
assert syntax_paths[0][0]=='incoming:wrapper'
assert all('Unchanged' in row[2] for row in context_paths)
assert all(ref in review_data['evidence'] for path in context_paths for ref in path[4])
journey_observation = {
 'execution':'recursive SQL control, NOT successful DuckPGQ traversal',
 'scope':'Captured neighborhood replay; not repository semantic completeness',
 'full_context_paths':context_paths,'syntax_only_paths':syntax_paths,
 'candidate_paths_hidden':len(context_paths)-len(syntax_paths),
 'architecture':{'components':len(review_data['components']),'alignment':'not_evaluated_no_policy'},
 'people':{'historical_candidates':len(review_data['people']),
           'github_logins_verified_in_capture':sum(p['github_login'] is not None for p in review_data['people']),
           'current_assignments':'not_refreshed','formal_ownership':'not_verified'},
 'ai_review':'not_run','run_head':review_data['head'],'source_evidence':len(review_data['evidence'])
}
print(json.dumps(journey_observation,indent=2))


{
  "execution": "recursive SQL control, NOT successful DuckPGQ traversal",
  "scope": "Captured neighborhood replay; not repository semantic completeness",
  "full_context_paths": [
    [
      "incoming:wrapper",
      "ExtractAggregationStats",
      "Unchanged body",
      1,
      [
        "native:wrapper"
      ]
    ],
    [
      "incoming:client",
      "Relation manager",
      "Unchanged",
      2,
      [
        "context:255",
        "native:wrapper"
      ]
    ],
    [
      "incoming:extractor",
      "Statistics extractor",
      "Unchanged",
      2,
      [
        "context:257",
        "native:wrapper"
      ]
    ]
  ],
  "syntax_only_paths": [
    [
      "incoming:wrapper",
      "ExtractAggregationStats",
      "Unchanged body",
      1,
      [
        "native:wrapper"
      ]
    ]
  ],
  "candidate_paths_hidden": 2,
  "architecture": {
    "components": 6,
    "alignment": "not_evaluated_no_policy"
  },
  "people": {
    "historical_candidates": 7,
    "gi

## Evaluation result: engine works; the named review app is not qualified yet

| Path | Observed result |
|---|---|
| Named Python `review-analysis` | Runs in configured root; prepared interpreter has DuckDB 1.5.5 |
| Named Python `review-duckdb2` | Runs with existing DuckDB 2.0.0 interpreter; all three captured extensions load; C++ symbol smoke yields 2 symbols |
| Named Deno `review-ui` | Saved with stable UUID, but MCP startup rejects its runtime slot before source execution |
| Named Python → notebook ports | Unavailable by live contract; `spur` is absent and the explicit publication probe catches NameError |
| DuckPGQ query through Python 2.0 | LOAD succeeds; both property-graph CREATE forms reject at PROPERTY. Root cause is not established here |
| Standalone Deno control | Redisplays the unchanged captured review HTML. This does not qualify named Deno or a native app |
| Native app doctor | Fails Level 0: no `spur-app.json`; DAG and port manifest are empty |

The captured snapshot contains **5 changed files, 21 evidence records, 6 components and 7 historical expertise candidates**. All 12 context-row evidence references resolve. A separate recursive SQL control retains the unchanged wrapper plus two unchanged clients. Hiding candidate edges removes the two external-client paths; this must not become “no impact.”

Architecture membership is available; alignment remains unevaluated without a versioned policy. People evidence is historical; current reviewer assignments and formal ownership are not established. AI review remains `not_run`. The C++ extractor still advertises `single_file` resolution.

Catalog Solve used `configuration.requires_any`: the seven local engine checks return **sat/pass**; the 13-gate named-app model returns **unsat/fail**, with six unqualified gates. These are supplied qualification facts, not a solver inspection of the runtime or proof of repository semantics. Remote solver artifact persistence hit its quota; exact requests/results are archived with this evaluation.

Recommended next slice: repair and qualify named Deno startup; qualify named Python publication and Deno asynchronous consumption; isolate the Python DuckPGQ parser path; then implement the captured-run native frontend and verify selection → unchanged context → exact evidence through the real DAG and app doctor. Add live collectors, architecture policy, current stakeholder enrichment and explicitly triggered AI review afterward.

This notebook deliberately retains a failed DuckPGQ cell and an unexecuted named Deno cell. It is a diagnostic notebook, not an all-green runnable app. Run `review-duckdb2` analysis before extension loading, then the parser diagnostics/recursive SQL control. No dependencies were installed, notebook defaults remain factory, and no app was packed.
